##Silver Transformation of the constructors bronze table
### 1. Read the table from bronze schema

### Getting the batch id as input parameter

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
%run "../00.Common/03.Helper_Notebook_Silver"

In [0]:
source_name = f"{catalog_name}.{bronze_schema}.constructors"
target_name = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
#import the sql function and filter the df with the batch id
from pyspark.sql import functions as F
constructors_df = (spark.table(source_name).filter(F.col("batch_id")==v_batch_id))
display(constructors_df)

### 2. Drop the unnecessary column (URL) 

In [0]:
constructors_dropped_df = constructors_df.drop("url")

### 3. Rename the columns

In [0]:
constructors_renamed_df = constructors_dropped_df.withColumnsRenamed({"constructorId":"constructor_id","name":"constructor_name"})
display(constructors_renamed_df)

### 4. Removing duplicates and NULL from the dataset

In [0]:
# Removing the null values using sql and column expressions
# circuits_clean_df = circuits_renamed_df.filter(
#     "circuit_id IS NOT NULL"
# )
# races_clean_df = races_renamed_df.filter(F.col("circuits_id").isNotNull())

In [0]:
# Removing duplicates based on the primary key
constructors_clean_df = constructors_renamed_df.dropDuplicates(["constructor_id"])
display(constructors_clean_df)

### 5. Transforming the column values 

In [0]:
# Converting the values to initcap format in locality and circuit name columns
constructors_final_df = (constructors_clean_df
                     .withColumn("nationality",F.initcap(F.col("nationality")))
)
display(constructors_final_df)

In [0]:
constructors_final_df.columns

### 6. Writing the final dataframe as table into the silver schema

In [0]:
write_to_silver(
    input_df = constructors_final_df,
    target_table = target_name,
    merge_condition = "t.constructor_id=s.constructor_id",
    columns_to_update = ['constructor_id',
 'constructor_name',
 'nationality',
 'ingestion_timestamp',
 'SourceFile',
 'batch_id']
)

In [0]:
%sql
select * from formula1_incr.silver.constructors;